# Geração de Log-Retornos Sintéticos com Modelos GARCH

**Autor:** Renan Santos Mendes

**Email:** renansantosmendes@gmail.com

**Disciplina:** Projeto em Deep Learning

**Curso:** Ciência de Dados e IA

---

## Objetivo

Nesta prática, vamos:

1. Coletar a série histórica de preços de uma ação usando a biblioteca
   `yfinance`.
2. Calcular a série de log-retornos a partir dos preços de fechamento.
3. Ajustar um modelo GARCH(1,1) sobre a série completa de log-retornos.
4. Simular trajetórias sintéticas de log-retorno para um horizonte futuro,
   usando o modelo ajustado.
5. Reconstruir o preço sintético a partir do último preço real conhecido.
6. Visualizar a série real seguida da continuação sintética, tanto em
   log-retorno quanto em preço.

Todos os dados sintéticos gerados nesta prática nunca existiram de fato,
mas são estatisticamente compatíveis com o processo estimado a partir dos
dados reais (mesma volatilidade condicional, mesma cauda pesada).

## 1. Instalação e importação das bibliotecas

Nesta célula, instalamos a biblioteca `arch` (usada para ajustar e simular
modelos GARCH) com o gerenciador de pacotes `uv`, e em seguida importamos
todas as bibliotecas utilizadas ao longo da prática.

In [ ]:
!uv pip install --system arch yfinance pgl-utils --quiet

In [ ]:
from __future__ import annotations

import os

import numpy as np
import pandas as pd
import yfinance as yf
from arch import arch_model
from arch.univariate import StudentsT

from pgl_utils.deep_learning import plot_real_vs_synthetic_continuation, plot_paths_grid

## 2. Configuração do experimento

Definimos aqui os parâmetros que controlam todo o experimento: o ticker da
ação, o intervalo de datas da série histórica, o horizonte de simulação, o
número de trajetórias sintéticas geradas e a semente aleatória usada para
garantir reprodutibilidade.

In [ ]:
RANDOM_SEED = 42
TICKER_SYMBOL = "AAPL"
START_DATE = "2020-01-01"
END_DATE = "2026-01-01"

SYNTHETIC_HORIZON_DAYS = 90
N_SYNTHETIC_PATHS = 200
N_PATHS_TO_PLOT = 30

N_GRID_PATHS = 9
GRID_NCOLS = 3
TAIL_REAL_DAYS_FOR_GRID = 60

OUTPUT_DIR = "outputs"

random_number_generator = np.random.default_rng(RANDOM_SEED)

## 3. Coleta dos dados históricos

A função abaixo baixa a série de preços de fechamento diário de um ativo,
em um intervalo de datas informado, usando a biblioteca `yfinance`.

In [ ]:
def fetch_price_series(
    ticker_symbol: str,
    start_date: str,
    end_date: str,
) -> pd.DataFrame:
    """Download the daily closing price series for a given ticker.

    Args:
        ticker_symbol: Ticker symbol of the asset to download (e.g.
            "AAPL").
        start_date: Start date of the historical series, in the format
            "YYYY-MM-DD".
        end_date: End date of the historical series, in the format
            "YYYY-MM-DD".

    Returns:
        A DataFrame indexed by date, with a single column named
        "close_price" containing the daily closing prices, sorted in
        ascending order of time and free of missing values.
    """
    raw_data = yf.download(
        ticker_symbol,
        start=start_date,
        end=end_date,
        progress=False,
    )
    price_data = raw_data[["Close"]].dropna()
    price_data.columns = ["close_price"]
    return price_data

In [ ]:
price_data = fetch_price_series(TICKER_SYMBOL, START_DATE, END_DATE)
price_data.tail()

## 4. Cálculo do log-retorno

O log-retorno entre dois instantes consecutivos é definido como a diferença
entre o logaritmo natural dos preços: $r_t = \ln(P_t) - \ln(P_{t-1})$. Essa
é a mesma série de log-retorno que será usada como entrada do modelo
GARCH.

In [ ]:
price_values = price_data["close_price"].to_numpy(dtype=np.float64)
price_index = price_data.index

log_prices = np.log(price_values)
log_returns = log_prices[1:] - log_prices[:-1]
return_dates = price_index[1:]

print(f"Total de log-retornos: {len(log_returns)}")
print(f"Período: {return_dates[0].date()} a {return_dates[-1].date()}")

## 5. Ajuste do modelo GARCH(1,1)

A função abaixo ajusta um modelo GARCH(1,1) com resíduos com distribuição
t-Student sobre a série de log-retornos, informada em porcentagem (prática
usual ao utilizar a biblioteca `arch`).

A semente aleatória é fixada diretamente na distribuição dos resíduos, pois
é esse objeto que controla a aleatoriedade utilizada posteriormente durante
a simulação de trajetórias sintéticas.

In [ ]:
def fit_garch(
    log_returns_pct: np.ndarray,
    random_seed: int,
):
    """Fit a GARCH(1,1) model with Student's t errors.

    Args:
        log_returns_pct: One-dimensional array with the log-return series,
            expressed in percentage points.
        random_seed: Seed used to initialize the random number generator
            of the Student's t error distribution, ensuring that later
            simulations are reproducible.

    Returns:
        The fitted result object returned by the `arch` library, which
        exposes methods such as `.forecast()` and `.summary()`.
    """
    model_specification = arch_model(
        log_returns_pct,
        mean="Constant",
        vol="Garch",
        p=1,
        q=1,
        dist="t",
    )
    model_specification.distribution = StudentsT(seed=random_seed)
    fitted_result = model_specification.fit(disp="off")
    print(fitted_result.summary())
    return fitted_result

In [ ]:
garch_result = fit_garch(log_returns * 100.0, RANDOM_SEED)

## 6. Simulação de trajetórias sintéticas de log-retorno

A partir do modelo GARCH ajustado, podemos simular múltiplas trajetórias
futuras de log-retorno. Cada trajetória é gerada respeitando a mesma
estrutura de volatilidade condicional e a mesma distribuição de cauda
pesada estimadas a partir dos dados reais.

In [ ]:
def simulate_synthetic_returns(
    garch_result,
    horizon_days: int,
    n_paths: int,
) -> np.ndarray:
    """Simulate synthetic log-return paths from a fitted GARCH model.

    Args:
        garch_result: Fitted GARCH result object, as returned by
            `fit_garch`.
        horizon_days: Number of future days to simulate for each path.
        n_paths: Number of independent synthetic paths to simulate.

    Returns:
        A two-dimensional array of shape (n_paths, horizon_days) with the
        simulated log-return paths, already converted back from
        percentage points to plain log-return units.
    """
    forecast_result = garch_result.forecast(
        horizon=horizon_days,
        method="simulation",
        simulations=n_paths,
        reindex=False,
    )
    synthetic_paths_pct = forecast_result.simulations.values[0]
    return synthetic_paths_pct / 100.0

In [ ]:
synthetic_returns = simulate_synthetic_returns(
    garch_result,
    SYNTHETIC_HORIZON_DAYS,
    N_SYNTHETIC_PATHS,
)

synthetic_dates = pd.bdate_range(
    start=return_dates[-1] + pd.Timedelta(days=1),
    periods=SYNTHETIC_HORIZON_DAYS,
)

synthetic_returns.shape

## 7. Reconstrução do preço sintético

Cada trajetória sintética de log-retorno pode ser transformada de volta em
uma trajetória de preço, encadeando os retornos a partir do último preço
real conhecido: $P_{\text{sintético}}[t+1] = P[t] \cdot e^{r[t+1]}$.

In [ ]:
def reconstruct_price_paths(
    last_known_price: float,
    synthetic_returns: np.ndarray,
) -> np.ndarray:
    """Reconstruct synthetic price paths from simulated log-returns.

    Args:
        last_known_price: Last observed real price, used as the starting
            point for every reconstructed path.
        synthetic_returns: Two-dimensional array of shape
            (n_paths, horizon_days) with the simulated log-return paths.

    Returns:
        A two-dimensional array with the same shape as
        `synthetic_returns`, containing the reconstructed synthetic price
        paths.
    """
    cumulative_log_return = np.cumsum(synthetic_returns, axis=1)
    return last_known_price * np.exp(cumulative_log_return)

In [ ]:
last_known_price = price_values[-1]
synthetic_prices = reconstruct_price_paths(last_known_price, synthetic_returns)

mean_synthetic_return_path = synthetic_returns.mean(axis=0)
mean_synthetic_price_path = synthetic_prices.mean(axis=0)

## 8. Exportação das trajetórias sintéticas

Salvamos as trajetórias sintéticas de log-retorno em um arquivo CSV, para
que possam ser reutilizadas em análises futuras sem precisar refazer a
simulação.

In [ ]:
os.makedirs(OUTPUT_DIR, exist_ok=True)

synthetic_returns_df = pd.DataFrame(
    synthetic_returns.T,
    index=synthetic_dates,
    columns=[f"path_{path_index}" for path_index in range(N_SYNTHETIC_PATHS)],
)
synthetic_returns_df.index.name = "date"

synthetic_csv_path = f"{OUTPUT_DIR}/garch_synthetic_returns.csv"
synthetic_returns_df.to_csv(synthetic_csv_path)

print(f"Trajetórias sintéticas salvas em: {synthetic_csv_path}")

## 9. Visualização: série real seguida da continuação sintética

O gráfico abaixo mostra, em dois painéis, a série real seguida diretamente
pelas trajetórias sintéticas simuladas, tanto em log-retorno quanto em
preço reconstruído.

In [ ]:
plot_real_vs_synthetic_continuation(
    return_dates=return_dates,
    log_returns=log_returns,
    price_index=price_index,
    price_values=price_values,
    synthetic_dates=synthetic_dates,
    synthetic_returns=synthetic_returns,
    synthetic_prices=synthetic_prices,
    mean_synthetic_return_path=mean_synthetic_return_path,
    mean_synthetic_price_path=mean_synthetic_price_path,
    ticker_symbol=TICKER_SYMBOL,
    n_paths_to_plot=N_PATHS_TO_PLOT,
    output_path=f"{OUTPUT_DIR}/garch_synthetic_continuation.png",
)

## 10. Visualização em grid: caminhos sintéticos individuais

Por fim, montamos um grid em que cada painel mostra uma única trajetória
sintética, precedida por um trecho recente da série real, o que facilita
a comparação visual caminho a caminho.

In [ ]:
grid_returns_path = f"{OUTPUT_DIR}/garch_synthetic_paths_grid_logreturn.png"

plot_paths_grid(
    real_tail_dates=return_dates[-TAIL_REAL_DAYS_FOR_GRID:],
    real_tail_values=log_returns[-TAIL_REAL_DAYS_FOR_GRID:],
    synthetic_dates=synthetic_dates,
    synthetic_paths=synthetic_returns,
    n_paths=N_GRID_PATHS,
    n_columns=GRID_NCOLS,
    title_prefix=f"{TICKER_SYMBOL} — Log-retorno: real + caminhos sintéticos individuais",
    y_axis_label="Log-retorno",
    output_path=grid_returns_path,
)

In [ ]:
grid_prices_path = f"{OUTPUT_DIR}/garch_synthetic_paths_grid_price.png"

plot_paths_grid(
    real_tail_dates=price_index[-TAIL_REAL_DAYS_FOR_GRID:],
    real_tail_values=price_values[-TAIL_REAL_DAYS_FOR_GRID:],
    synthetic_dates=synthetic_dates,
    synthetic_paths=synthetic_prices,
    n_paths=N_GRID_PATHS,
    n_columns=GRID_NCOLS,
    title_prefix=f"{TICKER_SYMBOL} — Preço: real + caminhos sintéticos individuais",
    y_axis_label="Preço",
    output_path=grid_prices_path,
    is_price=True,
)